In [ ]:
"""import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))"""

# Installing required python libraries

In [ ]:
!pip install duckdb --no-index --find-links=file:///kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/duck_pkg
!pip install polars[numpy,pandas,pyarrow] --no-index --find-links=file:///kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/polars_pkg
!pip install plotly
!pip install librosa
!pip install -U imbalanced-learn==0.12.4

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import duckdb as dd
import polars as pl
import pyarrow
import os
import glob
import shutil
import zipfile
import matplotlib.pyplot as plt
plt.style.use('dark_background')
import seaborn as sns
import plotly.express as px
import librosa
import pickle
from joblib import dump, load
from pathlib import Path
from IPython.display import Audio
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow_io as tfio
from tensorflow import keras

In [ ]:
tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
tf.tpu.experimental.initialize_tpu_system(tpu)
tpu_strategy = tf.distribute.TPUStrategy(tpu)

print("Number of accelerators: ", tpu_strategy.num_replicas_in_sync)

In [2]:
# Path to the directory containing your audio dataset
dataset_dir = '/kaggle/input/birdclef-2025/train_audio'
# Initialize an empty dictionary to store the mapping between audio files and labels
label_mapping = {}
# Iterate over subdirectories (classes) in the dataset directory
for label in os.listdir(dataset_dir):
    label_dir = os.path.join(dataset_dir, label)
    # Check if the item in the dataset directory is a directory
    if os.path.isdir(label_dir):
        # Iterate over audio files in the subdirectory (class)
        for audio_file in os.listdir(label_dir):
            # Add the mapping between audio file path and label to the dictionary
            audio_file_path = os.path.join(label_dir, audio_file)
            label_mapping[audio_file_path] = label
            
# label_mapping

# Create a list of tuples containing the audio file paths and labels
data = [(audio_file_path, label) for audio_file_path, label in label_mapping.items()]
# Create a Pandas DataFrame from the list of tuples
annotated_data = pd.DataFrame(data, columns=['audio_file_path', 'label'])

label_encoder = LabelEncoder()
annotated_data['encoded_label'] = label_encoder.fit_transform(annotated_data['label'])

print("dataset size: ",annotated_data.shape)

dataset size:  (28564, 3)


In [ ]:
annotated_data.head(5)

In [3]:
pl.Config(tbl_rows=200)
dd.sql("select * from \
    (select label, files, row_number()over(order by files desc) as rn from \
        (select label, count(distinct(audio_file_path)) as files from annotated_data group by label \
        )t1 \
    )t2 where rn <= 100").pl()

label,files,rn
str,i64,i64
"""grekis""",990,1
"""compau""",808,2
"""trokin""",787,3
"""roahaw""",709,4
"""banana""",610,5
"""whtdov""",572,6
"""socfly1""",543,7
"""yeofly1""",525,8
"""bobfly1""",514,9


In [4]:
def audio_waveframe(file_path):
    # Load the audio file
    audio_data, sampling_rate = librosa.load(file_path)
    # Calculate the duration of the audio file
    duration = len(audio_data) / sampling_rate
    # Create a time array for plotting
    time = np.arange(0, duration, 1/sampling_rate)
    # Plot the waveform
    plt.figure(figsize=(30, 4))
    plt.plot(time, audio_data, color='blue')
    plt.title('Audio Waveform')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plot = plt.show()
    return plot

def spectrogram(file_path):
    # Compute the short-time Fourier transform (STFT)
    n_fft = 500  # Number of FFT points 2048
    hop_length = 50  # Hop length for STFT 512
    audio_data, sampling_rate = librosa.load(file_path)
    stft = librosa.stft(audio_data, n_fft=n_fft, hop_length=hop_length)
    # Convert the magnitude spectrogram to decibels (log scale)
    spectrogram = librosa.amplitude_to_db(np.abs(stft))
    # Plot the spectrogram
    plt.figure(figsize=(30, 6))
    librosa.display.specshow(spectrogram, sr=sampling_rate, hop_length=hop_length, x_axis='time', y_axis='linear')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.tight_layout()
    plot = plt.show()
    return plot

def audio_analysis(file_path):
    aw = audio_waveframe(file_path)
    spg = spectrogram(file_path)
    return aw, spg

In [ ]:
audio_analysis('/kaggle/input/birdclef-2025/train_audio/1139490/CSA36385.ogg')
y, sr = librosa.load('/kaggle/input/birdclef-2025/train_audio/1139490/CSA36385.ogg')
Audio(y, rate=sr)

# Get the metadata

In [5]:
meta_data_schema = {
    "primary_label": pl.String,
    "secondary_labels": pl.String,
    "type": pl.String,
    "filename": pl.String,
    "collection": pl.String,
    "rating": pl.Float64,
    "url": pl.String,
    "latitude": pl.Float64,
    "longitude": pl.Float64,
    "scientific_name": pl.String,
    "common_name": pl.String,
    "author": pl.String,
    "license": pl.String
}

meta_data = pl.read_csv('/kaggle/input/birdclef-2025/train.csv',has_header=True, schema=meta_data_schema
                        , low_memory=True, null_values=["NA", "", "null","None"])

meta_data = meta_data.with_columns(
    pl.concat_str([pl.lit('/kaggle/input/birdclef-2025/train_audio/'),pl.col('filename')],separator='')\
                                   .alias('filename'))

#meta_data = meta_data.select(pl.exclude('filename'))

#meta_data = meta_data.rename({"full_file_path": "filename"})

meta_data.head(5)

primary_label,secondary_labels,type,filename,collection,rating,url,latitude,longitude,scientific_name,common_name,author,license
str,str,str,str,str,f64,str,f64,f64,str,str,str,str
"""1139490""","""['']""","""['']""","""/kaggle/input/birdclef-2025/tr…","""CSA""",0.0,"""http://colecciones.humboldt.or…",7.3206,-73.7128,"""Ragoniella pulchella""","""Ragoniella pulchella""","""Fabio A. Sarria-S""","""cc-by-nc-sa 4.0"""
"""1139490""","""['']""","""['']""","""/kaggle/input/birdclef-2025/tr…","""CSA""",0.0,"""http://colecciones.humboldt.or…",7.3206,-73.7128,"""Ragoniella pulchella""","""Ragoniella pulchella""","""Fabio A. Sarria-S""","""cc-by-nc-sa 4.0"""
"""1192948""","""['']""","""['']""","""/kaggle/input/birdclef-2025/tr…","""CSA""",0.0,"""http://colecciones.humboldt.or…",7.3791,-73.7313,"""Oxyprora surinamensis""","""Oxyprora surinamensis""","""Fabio A. Sarria-S""","""cc-by-nc-sa 4.0"""
"""1192948""","""['']""","""['']""","""/kaggle/input/birdclef-2025/tr…","""CSA""",0.0,"""http://colecciones.humboldt.or…",7.28,-73.8582,"""Oxyprora surinamensis""","""Oxyprora surinamensis""","""Fabio A. Sarria-S""","""cc-by-nc-sa 4.0"""
"""1192948""","""['']""","""['']""","""/kaggle/input/birdclef-2025/tr…","""CSA""",0.0,"""http://colecciones.humboldt.or…",7.3791,-73.7313,"""Oxyprora surinamensis""","""Oxyprora surinamensis""","""Fabio A. Sarria-S""","""cc-by-nc-sa 4.0"""


# Remove the recordings that have human voice

In [6]:
author_map = {
    'Alexandra Butrago-Cardona': 'Alexandra Buitrago-Cardona',
    'Ana María Ospina-Larrea | Daniela Murillo': 'Ana María Ospina-Larrea',
    'Diego A Gómez-Morales': 'Diego A. Gomez-Morales',
    'Eliana Barona- Cortés': 'Eliana Barona-Cortés',
    'Eliana Barona-Cortés | Daniela García-Cobos': 'Eliana Barona-Cortés',
    'Paula Caycedo-Rosales | Juan-Pablo López': 'Paula Caycedo-Rosales',
    'Fabio A. Sarria-S': 'Fabio A. Sarria-S'
}

sounds_to_exclude = meta_data.filter(
    pl.col("author").is_in (list(author_map.keys()))
)

In [7]:
dd.sql("select * from sounds_to_exclude where filename = '/kaggle/input/birdclef-2025/train_audio/1139490/CSA36385.ogg'").pl()

primary_label,secondary_labels,type,filename,collection,rating,url,latitude,longitude,scientific_name,common_name,author,license
str,str,str,str,str,f64,str,f64,f64,str,str,str,str
"""1139490""","""['']""","""['']""","""/kaggle/input/birdclef-2025/tr…","""CSA""",0.0,"""http://colecciones.humboldt.or…",7.3206,-73.7128,"""Ragoniella pulchella""","""Ragoniella pulchella""","""Fabio A. Sarria-S""","""cc-by-nc-sa 4.0"""


In [ ]:
dd.sql('select collection, count(distinct(filename)) as file_count from sounds_to_exclude group by collection order by 2 desc').pl()

In [ ]:
pl.Config(tbl_rows=200)
dd.sql("select * from (select author, file_count, row_number()over(order by file_count desc) rn \
from (select author, count(distinct(filename)) as file_count from sounds_to_exclude \
group by author)t1)t2 where rn <= 50 order by file_count desc"\
      ).pl()

In [ ]:
sounds_to_exclude.head(5)

In [8]:
final_train_df = dd.sql("select t1.* \
    from annotated_data t1 \
    left join sounds_to_exclude t2 \
    on t1.audio_file_path = t2.filename \
    where t2.author is null").pl()

# Arrive at the final training dataset

In [9]:
final_train_df.shape

(28523, 3)

In [ ]:
final_train_df.tail(5)

In [ ]:
#from tqdm import tqdm

labels = []
features = []

# Loop through each audio file in the dataset directory
for i in range(final_train_df.shape[0]):
    file_path = final_train_df.item(i,0)
    audio, sample_rate = librosa.load(file_path, sr=32000)
    samples_per_segment = sample_rate * 5
    if len(audio) > 7680000:
        total_samples = 7680000
    else:
        total_samples = len(audio)

    for j in range(0, total_samples+160000, samples_per_segment):
        if j + samples_per_segment <= total_samples:
            segment = audio[j:j + samples_per_segment]
            mfccs = librosa.feature.mfcc(y=segment, sr=32000, n_mfcc=40)
            flattened_features = (np.mean(mfccs.T, axis=0))
            features.append(flattened_features)
            labels.append(final_train_df.item(i,2))

In [ ]:
extracted_training_features_five_sec = np.array(features)
labels_five_sec = np.array(labels)

In [ ]:
with open("extracted_train_feat_five_sec_v5_2025", "wb") as file:   #Pickling
    pickle.dump(extracted_training_features_five_sec, file)
    
with open("labels_five_sec_v5_2025", "wb") as file:   #Pickling
    pickle.dump(labels_five_sec, file)

In [11]:
feature_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/extracted_train_feat_five_sec_v5_2025'
label_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/labels_five_sec_v5_2025'

with open(feature_file_path, "rb") as file:
    pickled_extracted_features_five_sec = pickle.load(file)
    
with open(label_file_path, "rb") as file:
    labels_five_sec = pickle.load(file)

In [12]:
x_five_sec = np.vstack(pickled_extracted_features_five_sec)
y_five_sec = labels_five_sec

print(x_five_sec.shape)
print(y_five_sec.shape)

(180142, 40)
(180142,)


In [13]:
x_train, x_test_val, y_train, y_test_val = train_test_split(x_five_sec, y_five_sec, test_size=0.4, random_state=42)
x_test, x_valid, y_test, y_valid = train_test_split(x_test_val, y_test_val, test_size=0.2, random_state=42)

C_range = np.logspace(-2, 10, 13)
gamma_range = np.logspace(-9, 3, 13)

print("Training data shape : {0}".format(x_train.shape))
print("Test data shape : {0}".format(x_test.shape))
print("Validation data shape : {0}".format(x_valid.shape))

Training data shape : (108085, 40)
Test data shape : (57645, 40)
Validation data shape : (14412, 40)


In [14]:
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

C_range = np.logspace(-2, 10, 13)
gamma_range = np.logspace(-9, 3, 13)
"""param_grid = dict(gamma=gamma_range, C=C_range, kernel=['rbf'])
cv = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_five_sec_scaled = scaler.fit_transform(x_five_sec)

grid = GridSearchCV(SVC(), param_grid=param_grid, cv=cv)
grid.fit(x_five_sec_scaled, y_five_sec)

print(
    "The best parameters are %s with a score of %0.2f"
    % (grid.best_params_, grid.best_score_)
)"""

'param_grid = dict(gamma=gamma_range, C=C_range, kernel=[\'rbf\'])\ncv = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)\n\nscaler = StandardScaler()\nx_five_sec_scaled = scaler.fit_transform(x_five_sec)\n\ngrid = GridSearchCV(SVC(), param_grid=param_grid, cv=cv)\ngrid.fit(x_five_sec_scaled, y_five_sec)\n\nprint(\n    "The best parameters are %s with a score of %0.2f"\n    % (grid.best_params_, grid.best_score_)\n)'

In [16]:
from sklearn.pipeline import Pipeline

rbf_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="rbf", C=C_range[5], gamma=gamma_range[4], probability=True, class_weight="balanced"))
])

In [ ]:
rbf_kernel_svm_model = rbf_kernel_svm_clf.fit(x_train, y_train)

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

rbf_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="rbf", C=C_range[5], gamma=gamma_range[4], probability=True, class_weight="balanced"))
])

sample_labels = [i for i in range(0, 182, 1)]
y_onehot_tests, y_scores = dict(), dict()

for fold, (train, test) in enumerate(skf.split(x_five_sec, y_five_sec)):
    print("fold = {0}".format(fold))
    x_train = np.ascontiguousarray(x_five_sec[train])
    y_train = np.ascontiguousarray(y_five_sec[train])
    x_test = np.ascontiguousarray(x_five_sec[test])
    y_test = np.ascontiguousarray(y_five_sec[test])
    print("train feature shape & label shape = {0} & {1}".format(x_train.shape, y_train.shape))
    print("test feature shape & label shape = {0} & {1}".format(x_test.shape, y_test.shape))
    rbf_kernel_svm_model = rbf_kernel_svm_clf.fit(x_train, y_train)
    y_predict = rbf_kernel_svm_model.predict_proba(x_test)
    print("predictions shape = {0}".format(y_predict.shape))
    ras_ovr = roc_auc_score(y_test, y_predict, multi_class='ovr', average='macro', labels=sample_labels)
    ras_ovo = roc_auc_score(y_test, y_predict, multi_class='ovo', average='macro', labels=sample_labels)
    print(f"Macro-averaged One-vs-Rest ROC AUC score: ", round(ras_ovr,2))
    print(f"Macro-averaged One-vs-One ROC AUC score: ", round(ras_ovo,2))
    y_scores[fold] = y_predict
    label_binarizer = LabelBinarizer().fit(y_train)
    y_onehot_test = label_binarizer.transform(y_test)
    y_onehot_tests[fold] = y_onehot_test